# Date et lieu de naissance, adresse : nos modèles internes face aux modèles d'Eden AI

**La question :** sur un champ manuscrit du bulletin (date et lieu de naissance, ou adresse du lieu de travail),
quel modèle lit le mieux : nos **modèles internes** (service LLM du SSP Cloud, aucune image ne sort de l'Insee)
ou les **modèles du marché** via **Eden AI** ? Avec quelle consigne, à quel coût, et l'un d'eux fait-il aussi bien
que le **PMQ** ?

**La méthode :** celle des notebooks 10, 12 et 13. La même image pour tous (le bloc de la question, découpé par
l'étape 1 du pipeline), la même consigne pour tous, jugées avec la **vérité arbitrée par le SeRN**, et comparées
par le test exact de **McNemar** sur les mêmes lignes.

**Les consignes de saisie du PMQ sont appliquées** : avant d'être jugée, chaque lecture est mise au format du
Fiqual (année complétée, 85 → 1985 ; jour sur 2 chiffres ; commune au libellé du dictionnaire ; département en
numéro ; pays vidé quand c'est la France), comme le fait le PMQ. Le tableau « effet des consignes » (cellule 9)
montre ce qui n'était pas une erreur de lecture mais une différence de convention.

Le code est dans le paquet (`mesure_qualite.decoupage` pour l'étape 1, `mesure_qualite.lecture` pour l'étape 2) ;
ce notebook ne fait que l'enchaîner et montrer les résultats. Les réglages des services et des sujets sont dans
`configs/lecture.yaml`.

## Comment lancer

1. **Cellule 1** : choisir le `SUJET` (`"naissance"` ou `"adresse_travail"`), laisser `TEST_RAPIDE = True`.
2. **Lancer tout, de haut en bas** (Run > Run All Cells). L'essai (cellule 7) écarte les modèles qui ne répondent
   pas, sans rien coûter de plus.
3. Si tout est bon : `TEST_RAPIDE = False` et relancer tout. **Les lectures déjà faites ne sont jamais refaites ni
   repayées** (cache), une cellule interrompue reprend là où elle s'est arrêtée.
4. **Lire** : cellule 9 (tableau principal), 11 (internes contre Eden AI), 12 (barre du PMQ), 14 (coût), 16 (phrases).
   Cellule 17 : la **page HTML** des erreurs (comparaison avec la vérité, découpe à côté de chaque lecture).
   Tout le détail part dans **MLflow** (cellule 18) : réglages, mesures, tableaux, page HTML, consignes, lectures.

| Cellule | Ce qu'elle fait | Coût Eden AI |
|---|---|---|
| 1 | Réglages | — |
| 2 – 3 | Fiqual et divergences du SeRN ; où est chaque champ dans le Fiqual | — |
| 4 | Étape 1 : découpe du bloc sur chaque bulletin (recalage sur le gabarit) | — |
| 5 | La vérité, bulletin par bulletin (groupes A, B, C) | — |
| 6 | Les lecteurs : modèles internes et modèles Eden AI | — |
| 7 | **Essai** : quelques bulletins par modèle | très faible |
| 8 | **Lectures** de tous les bulletins, chaque modèle avec chaque consigne | **principal** |
| 9 – 13 | Tableau principal, effet de la consigne, internes contre Eden AI, barre du PMQ, types d'erreurs | — |
| 14 | Temps et coût, projection annuelle | — |
| 15 – 16 | Excel ; phrases pour la réunion | — |
| 17 | **Page HTML** : comparaison avec la vérité, et chaque bulletin en erreur avec sa découpe | — |
| 18 | **MLflow** : tout le détail (serveur du projet et serveur personnel), ancien run remplacé | — |

In [ ]:
# 1. REGLAGES : tout ce qu'on change est ici (le reste du notebook ne se modifie pas)
import io
import logging
import re
from dataclasses import replace
from pathlib import Path

import mlflow
import pandas as pd
from IPython.display import display
from PIL import Image

from mesure_qualite import decoupage as dc
from mesure_qualite import lecture as lc
from mesure_qualite.decoupage import stockage
from mesure_qualite.decoupage.campagne import lire_configuration as config_campagnes
from mesure_qualite.decoupage.campagne import (
    parametres,
    racine_donnees,
    scans_du_formulaire,
)
from mesure_qualite.lecture import saisie

SUJET = "naissance"                 # "naissance" (question 2) ou "adresse_travail" (question 23)
CAMPAGNE = "RG25"
TEST_RAPIDE = True                  # True d'abord : 10 bulletins pour tout verifier ; puis False
N_RAPIDE = 10
N_ESSAI = 3                         # cellule 7 : bulletins lus par chaque modele avant de tout lancer
CONSIGNES = ["V0", "V1"]            # V0 simple, V1 detaillee (pieges connus) : texte en cellule 1, plus bas

# ---- Modeles internes (SSP Cloud, gratuits, aucune image ne sort de l'Insee) ----
MODELE_REFERENCE = "gemma4-26b-moe"  # la reference interne, a laquelle tout le monde est compare
MODELES_INTERNES = "auto"           # "auto" = tous les modeles du service qui lisent des images (essai cellule 7)
                                    # ou une liste, ex. ["gemma4-26b-moe", "qwen3-vl-32b"]

# ---- Modeles Eden AI (les images sortent de l'Insee : False = internes seuls) ----
UTILISER_EDEN = True
MODELES_EDEN = {                    # nom Eden AI -> pourquoi on le teste ; un nom faux est ecarte a l'essai
    "google/gemini-pro-latest":       "phare ferme : le meilleur sur les noms et prenoms (97,8 %)",
    "openai/gpt-latest":              "phare ferme d'OpenAI",
    "anthropic/claude-sonnet-latest": "rapide et economique",
    "google/gemini-flash-latest":     "rapide et economique",
    "mistral/pixtral-large-latest":   "ouvert, francais, installable un jour en interne",
    "qwen/qwen3-vl-plus":             "ouvert, plus gros que nos modeles internes",
}

# ---- Garde-fous ----
PLAFOND_DOLLARS = 30.0              # arret des lectures Eden AI au-dela de ce cout cumule
DUREE_MAX_MINUTES = 60              # par modele et par consigne ; relancer reprend la ou on s'est arrete
MAX_JETONS = 2000
DOCUMENTS_PAR_AN = 15 * 2450        # ordre de grandeur annuel du PMQ, pour projeter le cout

# ---- Consignes de saisie du PMQ : les lectures sont mises au format du Fiqual avant d'etre jugees ----
CONSIGNES_DE_SAISIE = True          # False = comparer les lectures brutes (pour voir l'effet des consignes)

# ---- Ou est le champ dans le Fiqual ? None = trouve tout seul (cellule 3) ----
COLONNES_FIQUAL = None              # ex. {"JOUR": 14, "MOIS": 15, "ANNEE": 16} si la cellule 3 se trompe

# ---- Ou sont les donnees et les resultats ----
RACINE = None                       # None = le bucket du projet ; sinon un dossier local de meme organisation
LECTURES = Path.home() / "work" / "banc-htr" / "resultats" / "lecture"   # caches des lectures (hors Git)
LECTURES.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")

S = lc.sujet(SUJET)
TEXTE = {v: lc.consigne(S, v) for v in CONSIGNES}
SUFFIXE = "_rapide" if TEST_RAPIDE else ""
print(f"{S.titre} | bloc {S.bloc} du {S.formulaire} | champs {S.noms_champs} | groupes {list(S.groupes)}")
print(f"campagne {CAMPAGNE} | {'TEST RAPIDE (' + str(N_RAPIDE) + ' bulletins)' if TEST_RAPIDE else 'COMPLET'} "
      f"| Eden AI {'oui' if UTILISER_EDEN else 'non'} | plafond {PLAFOND_DOLLARS} $")
print("\n--- consigne V1 ---\n" + TEXTE.get("V1", next(iter(TEXTE.values()))))

In [ ]:
# 2. LE FIQUAL (saisie du PMQ) ET LES DIVERGENCES ARBITREES PAR LE SeRN
CFG = config_campagnes()
CAMP = CFG["campagnes"][CAMPAGNE]
BASE = racine_donnees(CFG, RACINE)
DOSSIER = stockage.joindre(BASE, CAMP["racine"])

nom_fiqual = CAMP.get("fiqual")
fichiers_fiqual = ([stockage.joindre(DOSSIER, nom_fiqual)] if nom_fiqual else
                   [f for f in stockage.lister(DOSSIER, (".txt",)) if "FIQUAL" in Path(f).name.upper()])
fichiers_div = stockage.lister(stockage.joindre(DOSSIER, CAMP.get("divergences", "Divergences")), (".xlsx", ".xls"))
if not fichiers_fiqual or not all(stockage.existe(f) for f in fichiers_fiqual):
    raise RuntimeError(f"Fiqual introuvable sous {DOSSIER}")

FIQUAL = lc.lire_fiqual(fichiers_fiqual, S.type_ligne_fiqual)
DIV = lc.lire_divergences(fichiers_div)
DIV_SUJET = DIV[DIV.VARIABLE.map(S.champ_de_variable).notna()]
print(f"Fiqual : {len(FIQUAL)} bulletins | divergences : {len(DIV)} dont {len(DIV_SUJET)} sur ce sujet "
      f"({int(DIV_SUJET.arbitree.sum())} arbitrees par le SeRN)")
display(DIV_SUJET.groupby("VARIABLE").agg(divergences=("cle", "size"), arbitrees=("arbitree", "sum")))

In [ ]:
# 3. OU EST CHAQUE CHAMP DANS LE FIQUAL ?
#    Methode 1 (la plus sure) : pour chaque divergence, la colonne qui contient la VALEUR_PMQ de ce bulletin.
#    Methode 2 (date seulement, si les divergences ne suffisent pas) : trois colonnes voisines jour, mois, annee.
#    Reglage COLONNES_FIQUAL (cellule 1) : prioritaire.
COLS, VOTES = lc.colonnes_par_votes(FIQUAL, DIV, S)
SOURCE = {c: "divergences (votes)" for c in COLS}
if SUJET == "naissance" and not {"JOUR", "MOIS", "ANNEE"} <= set(COLS):
    annee_max = 2000 + int(CAMPAGNE[-2:])
    for c, i in lc.colonnes_date_par_contenu(FIQUAL, annee_max).items():
        if c not in COLS:
            COLS[c], SOURCE[c] = i, "contenu des colonnes (A VERIFIER)"
if "DEPARTEMENT" in S.noms_champs and "DEPARTEMENT" not in COLS and "COMMUNE" in COLS:
    i_dep = saisie.colonne_departement_par_contenu(FIQUAL, COLS["COMMUNE"], exclues=set(COLS.values()))
    if i_dep is not None:
        COLS["DEPARTEMENT"], SOURCE["DEPARTEMENT"] = i_dep, "contenu des colonnes (A VERIFIER)"
if COLONNES_FIQUAL:
    for c, i in COLONNES_FIQUAL.items():
        COLS[c], SOURCE[c] = int(i), "cellule 1 (COLONNES_FIQUAL)"
COLS = {c: COLS[c] for c in S.noms_champs if c in COLS}

if len(VOTES):
    display(VOTES)
lignes_fq = list(FIQUAL.values())
CARTE = pd.DataFrame([{
    "champ": c, "colonne": i, "trouvee par": SOURCE[c],
    "rempli_%": round(sum(bool(l[i].strip()) for l in lignes_fq if len(l) > i) / max(1, len(lignes_fq)) * 100, 1),
    "forme la plus frequente": pd.Series([lc.forme(l[i]) for l in lignes_fq if len(l) > i and l[i].strip()])
                               .value_counts().index[0] if lignes_fq else "",
} for c, i in COLS.items()])
display(CARTE)                      # forme : chiffres -> 9, lettres -> A (aucune valeur reelle affichee)
manquants = [c for c in S.noms_champs if c not in COLS]
if manquants:
    print(f"Champs non trouves dans le Fiqual : {manquants}. Ils sont lus, mais pas juges. "
          "Les renseigner dans COLONNES_FIQUAL (cellule 1) si besoin. Colonnes voisines de la commune "
          "(forme : chiffres -> 9, lettres -> A) :")
    centre = COLS.get("COMMUNE", max(COLS.values()))
    voisines = []
    for i in range(centre - 3, centre + 6):
        vals = [l[i].strip() for l in lignes_fq if len(l) > i and l[i].strip()]
        formes = pd.Series([lc.forme(v) for v in vals], dtype=str).value_counts()
        voisines.append({"colonne": i, "rempli_%": round(len(vals) / max(1, len(lignes_fq)) * 100, 1),
                         "formes": ", ".join(formes.index[:3]),
                         "role ici": next((c for c, j in COLS.items() if j == i), "")})
    display(pd.DataFrame(voisines))
if not COLS:
    raise RuntimeError("Aucun champ trouve dans le Fiqual : renseigner COLONNES_FIQUAL en cellule 1.")

In [ ]:
# 4. ETAPE 1 DU PIPELINE : le bloc de la question, decoupe sur chaque bulletin
#    Chaque scan est recale sur la page vierge (gabarit), puis le bloc est decoupe au meme endroit sur tous.
#    Les decoupes et leur index sont gardes dans le stockage : un bulletin deja decoupe n'est pas refait.
P = replace(parametres(CAMPAGNE, S.formulaire, CFG, racine=RACINE), blocs=(S.bloc,))
if not stockage.existe(stockage.joindre(P.gabarits, f"{P.gabarit}.json")):
    raise RuntimeError(f"Gabarit {P.gabarit} absent de {P.gabarits}. Le construire une fois depuis le terminal :\n"
                       f'  uv run python -m mesure_qualite.decoupage construire --pdf "<PDF vierge du {S.formulaire}>"')

SCANS = {lc.cle_image(s): s for s in scans_du_formulaire(P)}
CLES = sorted(k for k in SCANS if k in FIQUAL)
CLES = CLES[:N_RAPIDE] if TEST_RAPIDE else CLES
print(f"{len(SCANS)} scans {S.formulaire} | {len(CLES)} avec une ligne du Fiqual (retenus)")
INDEX = dc.decouper_campagne(P, scans=[SCANS[k] for k in CLES], paralleles=8)
INDEX = INDEX[(INDEX.bloc == S.bloc) & INDEX.scan.isin([SCANS[k] for k in CLES])].copy()
INDEX["cle"] = INDEX.scan.map(lc.cle_image)
print(dc.resume_index(INDEX))

DECOUPES = dict(zip(INDEX.cle, INDEX.decoupe))
IMAGES = {k: (lambda chemin=DECOUPES[k]: stockage.lire_octets(chemin)) for k in CLES if k in DECOUPES}
non_recales = INDEX[INDEX.methode != "recalage"]
if len(non_recales):
    print(f"{len(non_recales)} bulletins non recales (pris aux proportions) : a regarder")

# Controle visuel : trois decoupes (le bloc entier doit etre visible, libelles compris)
apercu = [Image.open(io.BytesIO(IMAGES[k]())) for k in list(IMAGES)[:3]]
if apercu:
    h = 220
    apercu = [im.resize((int(im.width * h / im.height), h)) for im in apercu]
    mosaique = Image.new("RGB", (sum(im.width for im in apercu) + 10 * len(apercu), h), "white")
    x = 0
    for im in apercu:
        mosaique.paste(im, (x, 0))
        x += im.width + 10
    display(mosaique)

In [ ]:
# 5. LA VERITE, bulletin par bulletin et champ par champ
#    A = pas de divergence (PMQ = PAD) : la valeur du PMQ ;  B = divergence arbitree : la valeur du SeRN ;
#    C = divergence non arbitree : verite inconnue, ligne ecartee du calcul.
VERITE = lc.table_verite(FIQUAL, DIV, S, COLS, list(IMAGES))
display(VERITE.groupby(["champ", "groupe"]).size().unstack(fill_value=0).reindex(columns=["A", "B", "C"], fill_value=0))

In [ ]:
# 6. LES LECTEURS : modeles internes (SSP Cloud) et modeles Eden AI
lc.cle_api("interne")
if MODELES_INTERNES == "auto":
    internes = lc.lister_internes()
    print(f"Modeles du service interne (hors embeddings, audio...) : {internes}")
else:
    internes = list(MODELES_INTERNES)
if MODELE_REFERENCE not in internes:
    internes.insert(0, MODELE_REFERENCE)
LECTEURS = [lc.Lecteur(m, "interne") for m in internes]
if UTILISER_EDEN:
    lc.cle_api("eden")
    LECTEURS += [lc.Lecteur(m, "eden") for m in MODELES_EDEN]
REFERENCE = lc.Lecteur(MODELE_REFERENCE, "interne")
COMPTEURS = lc.Compteurs()


def cache(lecteur, version):
    return lc.chemin_cache(str(LECTURES), CAMPAGNE, SUJET, lecteur, version)


def lire_avec(lecteur, version, images):
    paralleles = lc.lire_configuration()["fournisseurs"][lecteur.fournisseur].get("paralleles", 4)
    return lc.lire_tout(lecteur, version, TEXTE[version], S.noms_champs, images, cache(lecteur, version),
                        COMPTEURS, paralleles=paralleles, plafond_dollars=PLAFOND_DOLLARS,
                        duree_max_minutes=DUREE_MAX_MINUTES, max_jetons=MAX_JETONS)


display(pd.DataFrame([{"lecteur": l.nom, "fournisseur": l.fournisseur, "modele": l.modele,
                       "pourquoi": MODELES_EDEN.get(l.modele, "reference interne" if l == REFERENCE else "interne")}
                      for l in LECTEURS]))

In [ ]:
# 7. ESSAI : chaque modele lit N_ESSAI bulletins (consigne la plus longue). On garde ceux qui repondent
#    au bon format. Rien n'est gaspille : ces lectures sont dans le cache et ne seront pas refaites.
version_essai = CONSIGNES[-1]
essai_images = {k: IMAGES[k] for k in list(IMAGES)[:N_ESSAI]}
ESSAI, RETENUS = [], []
for lecteur in LECTEURS:
    lus, bilan = lire_avec(lecteur, version_essai, essai_images)
    json_ok = pd.to_numeric(lus.json_ok, errors="coerce").fillna(0).astype(bool)
    reussi = len(lus) == len(essai_images) and json_ok.all()
    cout = pd.to_numeric(lus.cout, errors="coerce").mean()
    ESSAI.append({"lecteur": lecteur.nom, "lus": len(lus), "json_ok": int(json_ok.sum()),
                  "s_par_appel": round(pd.to_numeric(lus.secondes, errors="coerce").mean(), 1),
                  "version_servie": lus.version.dropna().iloc[-1] if len(lus) else "",
                  "cout_estime_tout_$": round(cout * len(IMAGES) * len(CONSIGNES), 2) if pd.notna(cout) else None,
                  "retenu": reussi, "erreur": bilan.premiere_erreur[:120]})
    if reussi:
        RETENUS.append(lecteur)
ESSAI = pd.DataFrame(ESSAI)
display(ESSAI)
if REFERENCE not in RETENUS:
    raise RuntimeError(f"La reference {REFERENCE.nom} ne repond pas : verifier la cle du SSP Cloud et le nom du modele.")
print(f"Retenus : {[l.nom for l in RETENUS]} | adaptations (parametres refuses) : {COMPTEURS.adaptations or 'aucune'}")

In [ ]:
# 8. LECTURES de tous les bulletins : chaque modele retenu, avec chaque consigne (cache : rien n'est relu)
LUS, BILANS = {}, []
for lecteur in RETENUS:
    for version in CONSIGNES:
        lus, bilan = lire_avec(lecteur, version, IMAGES)
        LUS[f"{lecteur.nom} · {version}"] = (lecteur, version, lus)
        BILANS.append(bilan.__dict__)
BILANS = pd.DataFrame(BILANS)
display(BILANS)
incomplets = {n: len(l) for n, (_, _, l) in LUS.items() if len(l) < len(IMAGES)}
if incomplets:
    print(f"INCOMPLETS (relancer CETTE cellule : elle reprend la ou elle s'est arretee) : {incomplets}")
print(f"Cout cumule Eden AI de cette session : {COMPTEURS.cout:.2f} $")

In [ ]:
# 9. TABLEAU PRINCIPAL : chaque lecteur contre la verite arbitree, sur LES MEMES lignes
#    justes = meme valeur que la verite ; avouees = rien lu ou ILLISIBLE (part chez un humain) ;
#    silencieuses = fausses sans que personne le voie (les plus graves).
#    Les groupes (ex. DATE = jour, mois et annee tous justes) sont les chiffres a citer.
#    Consignes de saisie : chaque lecture est d'abord mise au format du Fiqual (85 -> 1985, ST -> SAINT,
#    5 -> 05, PARIS 18 -> PARIS 18E ARRONDISSEMENT...), comme le PMQ le fait, puis comparee.
COUVERTURE_MIN = 0.95
ANNEE_COLLECTE = 2000 + int(CAMPAGNE[-2:])
LONG_BRUT = {n: lc.en_long(l, S.noms_champs) for n, (_, _, l) in LUS.items() if len(l) >= COUVERTURE_MIN * len(IMAGES)}
LONG = ({n: saisie.au_format_fiqual(l, S, ANNEE_COLLECTE) for n, l in LONG_BRUT.items()}
        if CONSIGNES_DE_SAISIE else LONG_BRUT)
ecartes = sorted(set(LUS) - set(LONG))
if ecartes:
    print(f"Ecartes du tableau (moins de {COUVERTURE_MIN:.0%} des bulletins lus) : {ecartes}")
T = lc.juger(VERITE, LONG, S)
ORDRE = [*S.groupes, *[c for c in S.noms_champs if c in COLS]]
SYNTHESE = lc.resumer(T, list(LONG), [c for c in ORDRE if c in set(T.champ)])
principal = next(iter(S.groupes), ORDRE[0])
vue = SYNTHESE[SYNTHESE.champ == principal].sort_values(["justes_%", "silencieuses_%"], ascending=[False, True])
print(f"{principal} ({T[T.champ == principal].cle.nunique()} bulletins) :")
display(vue.reset_index(drop=True))
print("Tous les champs (justes_%) :")
display(SYNTHESE.pivot(index="lecteur", columns="champ", values="justes_%")[[c for c in ORDRE if c in set(SYNTHESE.champ)]])

# L'EFFET DES CONSIGNES : justes % sans puis avec la mise au format du Fiqual, champ par champ.
# Ce qui remonte n'etait pas une erreur de lecture, mais une difference de convention avec le PMQ.
T_BRUT = lc.juger(VERITE, LONG_BRUT, S)
S_BRUT = lc.resumer(T_BRUT, list(LONG_BRUT), [c for c in ORDRE if c in set(T_BRUT.champ)])
EFFET_CONSIGNES = (S_BRUT[S_BRUT.lecteur != "PMQ"][["champ", "lecteur", "justes_%"]]
                   .rename(columns={"justes_%": "sans consignes"})
                   .merge(SYNTHESE[["champ", "lecteur", "justes_%"]].rename(columns={"justes_%": "avec consignes"}),
                          on=["champ", "lecteur"]))
EFFET_CONSIGNES["gain (points)"] = (EFFET_CONSIGNES["avec consignes"] - EFFET_CONSIGNES["sans consignes"]).round(1)
print("Effet des consignes de saisie (justes %) :")
display(EFFET_CONSIGNES.pivot(index="lecteur", columns="champ", values="gain (points)"))

In [ ]:
# 10. EFFET DE LA CONSIGNE : pour chaque modele, V1 (detaillee) fait-elle mieux que V0 (simple) ?
#     Meme modele, memes lignes : seul le texte change.
EFFET = []
if {"V0", "V1"} <= set(CONSIGNES):
    for lecteur in RETENUS:
        a, b = f"{lecteur.nom} · V1", f"{lecteur.nom} · V0"
        if a in LONG and b in LONG:
            m = lc.comparer(T, a, b, [principal]).iloc[0]
            EFFET.append({"lecteur": lecteur.nom, "lignes": m.lignes, "V1 seule juste": m[f"{a} seul juste"],
                          "V0 seule juste": m[f"{b} seul juste"], "p": m.p,
                          "conclusion": m.conclusion.replace(a, "V1").replace(b, "V0")})
EFFET = pd.DataFrame(EFFET)
display(EFFET)
# Pour la suite, chaque modele garde SA meilleure consigne (justes sur le groupe principal)
MEILLEURE = {}
for lecteur in RETENUS:
    noms = [f"{lecteur.nom} · {v}" for v in CONSIGNES if f"{lecteur.nom} · {v}" in LONG]
    if noms:
        MEILLEURE[lecteur] = max(noms, key=lambda n: vue.set_index("lecteur")["justes_%"].get(n, -1))

In [ ]:
# 11. INTERNES CONTRE EDEN AI : chaque modele (sa meilleure consigne) contre la reference interne, memes lignes
ref = MEILLEURE.get(REFERENCE)
DUELS = []
for lecteur, nom in MEILLEURE.items():
    if nom == ref:
        continue
    m = lc.comparer(T, nom, ref, [principal]).iloc[0]
    j = vue.set_index("lecteur")["justes_%"]
    DUELS.append({"lecteur": nom, "fournisseur": lecteur.fournisseur, "justes_%": j.get(nom),
                  f"{REFERENCE.nom} justes_%": j.get(ref), "lui seul juste": m[f"{nom} seul juste"],
                  "reference seule juste": m[f"{ref} seul juste"], "p": m.p,
                  "conclusion": ("meilleur que la reference" if m.p < 0.05 and m[f"{nom} seul juste"] > m[f"{ref} seul juste"]
                                 else "moins bon que la reference" if m.p < 0.05 else "pas d'ecart etabli")})
DUELS = pd.DataFrame(DUELS)
if len(DUELS):
    display(DUELS.sort_values("justes_%", ascending=False).reset_index(drop=True))
meilleur_interne = max((n for l, n in MEILLEURE.items() if not l.externe), key=lambda n: vue.set_index("lecteur")["justes_%"][n])
externes = [n for l, n in MEILLEURE.items() if l.externe]
meilleur_eden = max(externes, key=lambda n: vue.set_index("lecteur")["justes_%"][n]) if externes else None
if meilleur_eden:
    m = lc.comparer(T, meilleur_eden, meilleur_interne, [principal]).iloc[0]
    print(f"Meilleur Eden AI : {meilleur_eden} | meilleur interne : {meilleur_interne} | "
          f"+{m[f'{meilleur_eden} seul juste']} / -{m[f'{meilleur_interne} seul juste']} | p = {m.p:.4f} -> "
          f"{'ecart reel' if m.p < 0.05 else 'pas d ecart etabli'}")

In [ ]:
# 12. LA BARRE DU PMQ : sa fourchette d'erreur, et ou se situe chaque lecteur (erreurs SILENCIEUSES)
BARRE = []
for champ in [c for c in ORDRE if c in set(T.champ)]:
    bas, haut = lc.barre_pmq(T, champ)
    for nom in MEILLEURE.values():
        s = SYNTHESE[(SYNTHESE.champ == champ) & (SYNTHESE.lecteur == nom)].iloc[0]
        e = s["silencieuses_%"] / 100
        BARRE.append({"champ": champ, "lecteur": nom, "justes_%": s["justes_%"], "silencieuses_%": s["silencieuses_%"],
                      "avouees_%": s["avouees_%"], "PMQ erreurs %": f"{bas * 100:.1f} a {haut * 100:.1f}",
                      "position": ("au niveau du PMQ ou mieux" if e <= bas else "dans la fourchette du PMQ"
                                   if e <= haut else "au-dessus du PMQ")})
BARRE = pd.DataFrame(BARRE)
display(BARRE[BARRE.champ == principal].sort_values("silencieuses_%").reset_index(drop=True))

In [ ]:
# 13. TYPES D'ERREURS (sans montrer de valeur) : chiffre voisin, lettre, rien lu...
TYPES = {}
for nom in MEILLEURE.values():
    e = T[T.champ.isin(S.noms_champs) & T.verite.notna() & (T[f"verdict {nom}"] != "juste")]
    TYPES[nom] = pd.Series([lc.type_erreur(l, v) for l, v in zip(e[f"lu {nom}"], e.verite)], dtype=str).value_counts()
TYPES = pd.DataFrame(TYPES).fillna(0).astype(int)
if len(TYPES):
    TYPES.loc["TOTAL"] = TYPES.sum()
display(TYPES)

In [ ]:
# 14. TEMPS ET COUT : par appel, pour 1 000 bulletins, et projete sur une annee de PMQ
COUTS = []
for lecteur, nom in MEILLEURE.items():
    lus = LUS[nom][2]
    sec = pd.to_numeric(lus.secondes, errors="coerce")
    cout = pd.to_numeric(lus.cout, errors="coerce")
    par_doc = cout.mean() if cout.notna().any() else (0.0 if not lecteur.externe else None)
    COUTS.append({"lecteur": nom, "fournisseur": lecteur.fournisseur, "s_par_bulletin": round(sec.mean(), 2),
                  "$_pour_1000": round(par_doc * 1000, 2) if par_doc is not None else None,
                  "$_par_an": round(par_doc * DOCUMENTS_PAR_AN) if par_doc is not None else None,
                  "images hors Insee": "oui" if lecteur.externe else "non"})
COUTS = pd.DataFrame(COUTS)
display(COUTS)
print(f"(projection : {DOCUMENTS_PAR_AN} bulletins par an ; les modeles internes ne coutent que le calcul du SSP Cloud)")

In [ ]:
# 15. EXCEL DETAILLE (un onglet par tableau)
f_xlsx = LECTURES / f"lecture_{SUJET}_{CAMPAGNE}{SUFFIXE}.xlsx"
REGLAGES = pd.DataFrame([
    ("sujet", SUJET), ("campagne", CAMPAGNE), ("test_rapide", TEST_RAPIDE), ("bulletins", len(IMAGES)),
    ("formulaire", S.formulaire), ("bloc", S.bloc), ("gabarit", P.gabarit), ("consignes", ", ".join(CONSIGNES)),
    ("reference", REFERENCE.nom), ("modeles internes", MODELES_INTERNES if isinstance(MODELES_INTERNES, str)
                                   else ", ".join(MODELES_INTERNES)),
    ("Eden AI", UTILISER_EDEN), ("modeles Eden AI", ", ".join(MODELES_EDEN) if UTILISER_EDEN else "-"),
    ("retenus a l'essai", ", ".join(l.nom for l in RETENUS)), ("plafond $", PLAFOND_DOLLARS),
    ("duree max min", DUREE_MAX_MINUTES), ("max jetons", MAX_JETONS), ("colonnes Fiqual", str(COLS)),
    ("colonnes trouvees par", str(SOURCE)), ("parametres refuses", str(COMPTEURS.adaptations)),
    ("consignes de saisie appliquees", CONSIGNES_DE_SAISIE),
], columns=["reglage", "valeur"])
TABLEAUX = {"1_principal": vue, "2_synthese": SYNTHESE, "3_consigne": EFFET, "4_duels": DUELS,
            "5_barre_pmq": BARRE, "6_types_erreurs": TYPES.reset_index(names="type"), "7_couts": COUTS,
            "8_essai": ESSAI, "9_bilans_lectures": BILANS, "10_colonnes_fiqual": CARTE,
            "11_votes_colonnes": VOTES, "12_reglages": REGLAGES, "13_effet_consignes": EFFET_CONSIGNES}
with pd.ExcelWriter(f_xlsx, engine="openpyxl") as w:
    for onglet, df in TABLEAUX.items():
        df.to_excel(w, sheet_name=onglet, index=False)
print(f"Excel : {f_xlsx}")

In [ ]:
# 16. RESUME (calcule a partir des tableaux ci-dessus ; enregistre dans MLflow : resume.txt)
j = vue.set_index("lecteur")
pmq = j.loc["PMQ"]
quoi = (f"{principal.lower()} : {', '.join(S.groupes[principal]).lower()} tous justes" if principal in S.groupes
        else principal.lower())
PHRASES = [
    f"Sur {int(j.loc[ref, 'lignes'])} bulletins ({quoi}), le PMQ est juste a {pmq['justes_%']} % {pmq['IC95']}.",
    (f"Notre reference interne {ref} : {j.loc[ref, 'justes_%']} % {j.loc[ref, 'IC95']}, "
     f"dont {j.loc[ref, 'silencieuses_%']} % d'erreurs silencieuses."),
    f"Meilleur modele interne : {meilleur_interne} ({j.loc[meilleur_interne, 'justes_%']} %).",
]
if meilleur_eden:
    m = lc.comparer(T, meilleur_eden, meilleur_interne, [principal]).iloc[0]
    ecart = "un ecart reel" if m.p < 0.05 else "pas d'ecart etabli"
    PHRASES.append(f"Meilleur modele Eden AI : {meilleur_eden} ({j.loc[meilleur_eden, 'justes_%']} %) : "
                   f"+{m[f'{meilleur_eden} seul juste']} / -{m[f'{meilleur_interne} seul juste']} bulletins face au "
                   f"meilleur interne, p = {m.p:.4f}, {ecart}.")
    c = COUTS.set_index("lecteur")
    if meilleur_eden in c.index and pd.notna(c.loc[meilleur_eden, "$_pour_1000"]):
        PHRASES.append(f"Son cout : {c.loc[meilleur_eden, '$_pour_1000']} $ pour 1 000 bulletins, environ "
                       f"{c.loc[meilleur_eden, '$_par_an']} $ par an ; les images sortent alors de l'Insee.")
if TEST_RAPIDE:
    PHRASES.append("TEST RAPIDE : chiffres sur quelques bulletins seulement, a ne pas citer.")
for k, phrase in enumerate(PHRASES, 1):
    print(f"{k}. {phrase}")

In [ ]:
# 17. PAGE HTML : COMPARAISON AVEC LA VERITE ET ERREURS, bulletin par bulletin, decoupe a cote
#     En haut : par champ et par lecteur, justes / fausses / rien lu contre la verite.
#     Puis chaque bulletin ou un lecteur (ou le PMQ) se trompe : la decoupe, la verite et sa source
#     (PMQ = PAD, ou arbitrage du SeRN), ce qu'a lu chaque lecteur, et le verdict ecrit en clair.
#     Donnees reelles : la page reste dans le stockage du projet et dans MLflow, jamais sur Git.
LECTEURS_PAGE = list(MEILLEURE.values())          # chaque modele avec sa meilleure consigne
COMPARAISON = lc.comparaison_verite(T, LECTEURS_PAGE)
PAGE_ERREURS = lc.galerie_erreurs(T, LECTEURS_PAGE, IMAGES, S,
                                  titre=f"campagne {CAMPAGNE}, {len(IMAGES)} bulletins{' (test rapide)' if TEST_RAPIDE else ''}")
f_html = LECTURES / f"erreurs_{SUJET}_{CAMPAGNE}{SUFFIXE}.html"
f_html.write_text(PAGE_ERREURS, encoding="utf-8")
display(COMPARAISON.pivot(index="lecteur", columns="champ", values="justes_%"))
print(f"Page des erreurs : {f_html} ({len(PAGE_ERREURS) / 1e6:.1f} Mo) : l'ouvrir depuis le panneau de gauche, "
      "clic droit > Open in New Browser Tab")

In [ ]:
# 18. MLflow : TOUT le detail, sur le serveur du projet et/ou le serveur personnel
#     - etiquettes : etude, notebook, sujet, campagne, commit Git (et si le code n'etait pas commite) ;
#     - parametres : tous les reglages, colonnes du Fiqual, modeles essayes, retenus, ecartes (et pourquoi),
#       versions reellement servies, parametres refuses par certains modeles ;
#     - mesures : par champ et par lecteur (justes, intervalle de confiance, avouees, silencieuses), barre du PMQ,
#       McNemar (consigne, duels), temps et cout ;
#     - fichiers : la page HTML des erreurs (comparaison avec la verite, decoupes), l'Excel, chaque tableau
#       en CSV, les consignes, la configuration, le resume, le journal ;
#       si ENREGISTRER_TOUT : aussi la table ligne par ligne, les lectures brutes et l'apercu des decoupes
#       (donnees reelles : elles restent dans le stockage du projet, jamais sur Git).
import json
import subprocess
import tempfile

from mlflow.tracking import MlflowClient

from mesure_qualite.evaluation import double_mlflow as dm

SERVEURS_MLFLOW = ["projet", "personnel"]   # ["personnel"] = votre serveur seul ; ["projet"] = le projet seul
ENREGISTRER_TOUT = True
REMPLACER_ANCIENS = False   # True = les anciens runs du meme sujet et de la meme campagne passent en statut "remplace"
NOM_NOTEBOOK = "lecture_modeles_internes_et_eden"


def nom_mlflow(*morceaux):
    return re.sub(r"[^a-z0-9_.-]+", "_", "_".join(str(m) for m in morceaux).lower()).strip("_")[:240]


def git(*args):
    try:
        return subprocess.run(["git", *args], capture_output=True, text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return ""


def mesures():
    m = {"bulletins": len(IMAGES), "cout_eden_dollars": round(COMPTEURS.cout, 4),
         "recalage_pct": float((INDEX.methode == "recalage").mean() * 100)}
    for _, r in SYNTHESE.iterrows():
        bas, haut = lc.wilson(int(r["justes"]), int(r["lignes"]))
        base = nom_mlflow(r["champ"], r["lecteur"])
        m.update({f"{base}.justes_pct": r["justes_%"], f"{base}.ic_bas_pct": round(bas * 100, 2),
                  f"{base}.ic_haut_pct": round(haut * 100, 2), f"{base}.avouees_pct": r["avouees_%"],
                  f"{base}.silencieuses_pct": r["silencieuses_%"], f"{base}.lignes": r["lignes"]})
    for champ in set(BARRE.champ):
        bas, haut = lc.barre_pmq(T, champ)
        m.update({f"{nom_mlflow(champ)}.pmq_erreurs_min_pct": round(bas * 100, 2),
                  f"{nom_mlflow(champ)}.pmq_erreurs_max_pct": round(haut * 100, 2)})
    for _, r in EFFET.iterrows():
        m[f"consigne_v1_contre_v0.{nom_mlflow(r['lecteur'])}.p"] = r["p"]
    for _, r in DUELS.iterrows():
        m[f"contre_reference.{nom_mlflow(r['lecteur'])}.p"] = r["p"]
    for _, r in COUTS.iterrows():
        base = nom_mlflow(r["lecteur"])
        for k, col in [("s_par_bulletin", "s_par_bulletin"), ("dollars_1000", "$_pour_1000"),
                       ("dollars_an", "$_par_an")]:
            if pd.notna(r[col]):
                m[f"{base}.{k}"] = r[col]
    return {k: float(v) for k, v in m.items() if pd.notna(v)}


def enregistrer():
    with mlflow.start_run(run_name=f"{SUJET}-{CAMPAGNE}{SUFFIXE}") as run:
        mlflow.set_tags({"etude": f"lecture-{SUJET.replace('_', '-')}", "notebook": NOM_NOTEBOOK,
                         "statut": "exploratoire", "sujet": SUJET, "campagne": CAMPAGNE,
                         "test_rapide": str(TEST_RAPIDE), "git_commit": git("rev-parse", "--short", "HEAD") or "inconnu",
                         "git_branche": git("branch", "--show-current"),
                         "git_code_modifie": "oui" if git("status", "--porcelain") else "non"})
        mlflow.log_params({nom_mlflow(r.reglage): str(r.valeur)[:500] for r in REGLAGES.itertuples()})
        mlflow.log_params({
            "modeles_essayes": ", ".join(l.nom for l in LECTEURS)[:500],
            "modeles_ecartes": json.dumps({e.lecteur: e.erreur or "reponse hors format"
                                           for e in ESSAI[~ESSAI.retenu].itertuples()}, ensure_ascii=False)[:500],
            "versions_servies": json.dumps(dict(zip(ESSAI.lecteur, ESSAI.version_servie)), ensure_ascii=False)[:500],
        })
        mlflow.log_metrics(mesures())
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            for onglet, df in TABLEAUX.items():
                df.to_csv(d / f"{onglet}.csv", index=False)
            mlflow.log_artifacts(str(d), "tableaux")
        mlflow.log_artifact(str(f_xlsx))
        for v, texte in TEXTE.items():
            mlflow.log_text(texte, f"consignes/{v}.txt")
        mlflow.log_text((lc.sujets.racine_depot() / "configs" / "lecture.yaml").read_text(encoding="utf-8"),
                        "configuration/lecture.yaml")
        mlflow.log_text(PAGE_ERREURS, "erreurs.html")
        mlflow.log_text(COMPARAISON.to_csv(index=False), "tableaux/14_comparaison_verite.csv")
        mlflow.log_text("\n".join(PHRASES), "resume.txt")
        mlflow.log_text(JOURNAL_DECISIONS, "journal_decisions.md")
        if ENREGISTRER_TOUT:
            with tempfile.TemporaryDirectory() as d:
                d = Path(d)
                T.to_csv(d / "table_ligne_par_ligne.csv", index=False)
                INDEX.to_csv(d / "index_decoupes.csv", index=False)
                for nom, (_, _, lus) in LUS.items():
                    lus.to_csv(d / f"lectures_{nom_mlflow(nom)}.csv", index=False)
                if apercu:
                    mosaique.save(d / "apercu_decoupes.png")
                mlflow.log_artifacts(str(d), "detail")
        if REMPLACER_ANCIENS:
            remplacer_anciens(run)
        return run.info.run_id


def remplacer_anciens(run):
    # Les runs precedents du meme sujet, de la meme campagne et du meme mode passent en
    # statut "remplace" (ils restent consultables) ; le nouveau garde "exploratoire".
    client = MlflowClient()
    filtre = (f"tags.etude = 'lecture-{SUJET.replace('_', '-')}' and tags.sujet = '{SUJET}' "
              f"and tags.campagne = '{CAMPAGNE}' and tags.test_rapide = '{TEST_RAPIDE}'")
    anciens = client.search_runs([run.info.experiment_id], filtre, max_results=200)
    for ancien in anciens:
        if ancien.info.run_id != run.info.run_id and ancien.data.tags.get("statut") != "remplace":
            client.set_tag(ancien.info.run_id, "statut", "remplace")
            client.set_tag(ancien.info.run_id, "remplace_par", run.info.run_id)
            print(f"  ancien run {ancien.info.run_id[:8]} -> statut remplace")


JOURNAL_DECISIONS = '''| Decision | Pourquoi |
|---|---|
| Meme image pour tous : le bloc de la question, decoupe par l'etape 1 | seul le lecteur change |
| Meme consigne pour tous, deux versions testees | separer l'effet du modele de celui de la consigne |
| Verite arbitree par le SeRN, divergences non arbitrees ecartees | le Fiqual seul est la saisie du PMQ |
| Memes lignes pour tous | McNemar compare des paires |
| Normalisation des deux cotes (zeros, accents, abreviations) | ne pas compter faux 05 pour 5 |
| Internes d'abord, Eden AI en option | avec Eden AI, les images sortent de l'Insee |
| Cache, plafond de depense, duree maximale | ne jamais repayer, ne jamais depasser le budget |'''

RUNS = dm.partout(enregistrer, perso="personnel" in SERVEURS_MLFLOW, partage="projet" in SERVEURS_MLFLOW)
print(RUNS)

## Comment lire et conclure

1. **Cellule 3, les colonnes du Fiqual.** « divergences (votes) » : fiable. « contenu des colonnes (À VÉRIFIER) » :
   vérifier l'onglet `9_colonnes` de l'Excel avant de citer un chiffre.
2. **Cellule 4, l'aperçu.** Le bloc entier doit être visible, libellés compris. Les bulletins « non recalés »
   sont comptés : s'il y en a beaucoup, regarder leur formulaire.
3. **Cellule 9.** Le chiffre à citer est celui du **groupe** (ex. DATE : jour, mois et année tous justes), contre la
   vérité arbitrée, avec son intervalle de confiance. Regarder aussi les **silencieuses** : ce sont elles qui coûtent.
4. **Cellules 10 et 11.** « pas d'écart établi » veut dire qu'on ne peut pas conclure qu'un modèle est meilleur :
   le dire ainsi, sans arrondir dans un sens ou dans l'autre.
5. **Cellule 12.** Un lecteur dont les erreurs silencieuses sont sous la barre basse du PMQ fait au moins aussi bien
   que lui sur ces lignes.

## Journal des décisions

| Décision | Pourquoi |
|---|---|
| Même image pour tous : le bloc de la question, découpé par l'étape 1 | seul le lecteur change : comparaison équitable |
| Même consigne pour tous, deux versions testées | séparer l'effet du modèle de l'effet de la consigne |
| Vérité arbitrée par le SeRN, divergences non arbitrées écartées | le Fiqual seul est la saisie du PMQ, qui se trompe aussi |
| Mêmes lignes pour tous (lues par chaque lecteur) | McNemar compare des paires : il faut les mêmes bulletins |
| Valeurs comparées après normalisation des deux côtés (zéros, accents, abréviations) | ne pas compter faux « 05 » pour « 5 » ni « BD » pour « BOULEVARD » |
| Les modèles internes d'abord, Eden AI en option (`UTILISER_EDEN`) | avec Eden AI, les images sortent de l'Insee |
| Cache des lectures, plafond de dépense, durée maximale | ne jamais repayer, ne jamais dépasser le budget |